## Supporting notebook to prepare hydrodynamic data for notebook 0213

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from opentnsim.graph.mixins import FIS as fis
import xarray as xr
import networkx as nx
from shapely import transform
from shapely.geometry import Point
import geopandas as gpd
from opentnsim.environment.utils import (create_default_hydrodynamic_dataset, 
                                         add_specific_environmental_data_on_node, 
                                         interpolate_data_on_route,
                                         add_closest_node_to_xr_dataset)
from pyproj import CRS
%matplotlib inline

#### Loading raw data

In [2]:
path = os.getcwd()

In [3]:
df = pd.read_csv(os.path.join(path,"water_level_data_IJmuiden.csv"), delimiter=';', encoding='ISO-8859-1', on_bad_lines='skip', low_memory=False)

#### Preparing time data

In [4]:
df['TIME'] = df['WAARNEMINGDATUM'] + ' ' + df['WAARNEMINGTIJD']
df['TIME'] = pd.to_datetime(df['TIME'], format = "%d-%m-%Y %H:%M:%S")

#### Selecting columns

In [5]:
df = df[['TIME','MEETPUNT_IDENTIFICATIE','NUMERIEKEWAARDE','GROOTHEID_OMSCHRIJVING','BEMONSTERINGSHOOGTE','LAT','LON','EPSG']]

#### Translating data

In [6]:
df = df.rename(columns={'MEETPUNT_IDENTIFICATIE':'Location',
                        'NUMERIEKEWAARDE':'Value',
                        'GROOTHEID_OMSCHRIJVING':'Physical quantity',
                        'BEMONSTERINGSHOOGTE':'Measurement height'})
df.loc[df['Physical quantity'] == 'Waterhoogte', 'Physical quantity'] = 'Water level'

#### Preparing data

In [7]:
#cm to m
df.loc[df['Physical quantity'] == 'Water level', 'Value'] = df.loc[df['Physical quantity'] == 'Water level', 'Value']/100
df['Measurement height'] = df['Measurement height']/100

In [8]:
df = df.loc[df['Measurement height'].dropna().index]
df = df.sort_values('TIME')
df = df.reset_index(drop=True)

In [9]:
# Initialize a list to store cleaned indices
to_drop = []

for _, df_group in df.groupby(['Location','Physical quantity','Measurement height']):
    # Sort by TIME
    df_group = df_group.sort_values('TIME')

    # Identify duplicate TIME indices (keep the first)
    duplicated_times = df_group['TIME'][df_group['TIME'].duplicated(keep='first')]
    to_drop.extend(duplicated_times.index.tolist())

    # Remove duplicates for interpolation
    df_group_unique = df_group[~df_group['TIME'].duplicated(keep='first')]

    # Save mapping from TIME to original integer indexes
    time_to_idx = df_group_unique.index.to_series()

    # Set TIME as index for time-based interpolation
    df_group_unique = df_group_unique.set_index('TIME')

    # Interpolate
    interpolated = df_group_unique['Value'].interpolate(method='time').ffill().bfill()

    # Assign back using original integer indexes
    df.loc[time_to_idx, 'Value'] = interpolated.values

# Drop duplicates from the original df
df = df.drop(index=to_drop)
df = df.reset_index(drop=True)

In [10]:
df

,TIME,Location,Value,Physical quantity,Measurement height,LAT,LON,EPSG
0,2026-04-30 23:00:00,"IJmuiden, noordersluis, oost",-0.40,Water level,0.0,52.4672,4.614,ETRS89
1,2026-04-30 23:00:00,"IJmuiden, noordersluis, west",-0.97,Water level,0.0,52.4678,4.604,ETRS89
2,2026-04-30 23:10:00,"IJmuiden, noordersluis, west",-0.97,Water level,0.0,52.4678,4.604,ETRS89
3,2026-04-30 23:10:00,"IJmuiden, noordersluis, oost",-0.40,Water level,0.0,52.4672,4.614,ETRS89
4,2026-04-30 23:20:00,"IJmuiden, noordersluis, oost",-0.41,Water level,0.0,52.4672,4.614,ETRS89
...,...,...,...,...,...,...,...,...
8925,2026-05-31 22:40:00,"IJmuiden, noordersluis, oost",-0.41,Water level,0.0,52.4672,4.614,ETRS89
8926,2026-05-31 22:50:00,"IJmuiden, noordersluis, west",-0.54,Water level,0.0,52.4678,4.604,ETRS89
8927,2026-05-31 22:50:00,"IJmuiden, noordersluis, oost",-0.42,Water level,0.0,52.4672,4.614,ETRS89
8928,2026-05-31 23:00:00,"IJmuiden, noordersluis, oost",-0.43,Water level,0.0,52.4672,4.614,ETRS89


#### Creating xr.DataSet

In [11]:
hydrodynamic_data = xr.Dataset()

In [12]:
# Ensure datetime
df['TIME'] = pd.to_datetime(df['TIME'], errors='coerce')
df = df[~df['TIME'].isna()]

# Define global time axis
t_min = df['TIME'].min()
t_max = df['TIME'].max()

dt = '10min'  # or '1H', '5min', etc.

common_time = pd.date_range(start=t_min, end=t_max, freq=dt)

In [13]:
# Global station list
all_stations = df['Location'].unique()

# Global time axis
t_min = df['TIME'].min()
t_max = df['TIME'].max()
common_time = pd.date_range(start=t_min, end=t_max, freq=dt)

hydrodynamic_data = xr.Dataset()

# --- Loop over quantities ---
for quantity, df_quantity in df.groupby('Physical quantity'):

    stations = []
    xs = []
    ys = []
    epsgs = []
    aligned_data = []

    for station in all_stations:

        df_location = df_quantity[df_quantity['Location'] == station]

        # --- Metadata (from full df) ---
        df_meta = df[df['Location'] == station]

        lat = df_meta['LAT'].mode().iloc[0] if 'LAT' in df_meta else np.nan
        lon = df_meta['LON'].mode().iloc[0] if 'LON' in df_meta else np.nan
        epsg_val = df_meta['EPSG'].mode().iloc[0] if 'EPSG' in df_meta else None

        try:
            crs = CRS.from_user_input(epsg_val)
            epsg_val = f"EPSG:{crs.to_epsg()}"
        except:
            pass

        stations.append(station)
        xs.append(lat)
        ys.append(lon)
        epsgs.append(epsg_val)

        # --- If no data for this station ---
        if df_location.empty:
            series = pd.Series(index=common_time, dtype=float, name=station)
            aligned_data.append(series)
            continue

        # --- Clean data ---
        df_loc = df_location.sort_values('TIME')

        # Remove duplicate timestamps
        df_loc = df_loc.drop_duplicates(subset='TIME')

        # Average over measurement height
        df_loc = df_loc.groupby('TIME')['Value'].mean()

        # --- Reindex to common time ---
        df_loc = df_loc.reindex(common_time)

        # --- Interpolate + fill ---
        df_loc = (
            df_loc
            .interpolate(method='time')
            .ffill()
            .bfill()
        )

        df_loc.name = station
        aligned_data.append(df_loc)

    # --- Combine stations ---
    df_aligned = pd.concat(aligned_data, axis=1)

    # --- Convert to xarray ---
    data = df_aligned.to_numpy().T
    time = common_time.to_numpy()

    da = xr.DataArray(
        data,
        dims=('STATION', 'TIME'),
        coords={
            'STATION': stations,
            'TIME': time,
            'LAT': ("STATION", xs),
            'LON': ("STATION", ys),
            'EPSG': ("STATION", epsgs)
        },
        name=quantity
    )

    hydrodynamic_data[quantity] = da

In [15]:
hydrodynamic_data

<xarray.Dataset> Size: 107kB
Dimensions:      (STATION: 2, TIME: 4465)
Coordinates:
  * STATION      (STATION) <U28 224B 'IJmuiden, noordersluis, oost' 'IJmuiden...
    LAT          (STATION) float64 16B 52.47 52.47
    LON          (STATION) float64 16B 4.614 4.604
    EPSG         (STATION) <U9 72B 'EPSG:4258' 'EPSG:4258'
  * TIME         (TIME) datetime64[us] 36kB 2026-04-30T23:00:00 ... 2026-05-3...
Data variables:
    Water level  (STATION, TIME) float64 71kB -0.4 -0.4 -0.41 ... -0.54 -0.57

In [16]:
hydrodynamic_data = hydrodynamic_data.assign_coords(
    STATION=[
        "1",
        "0",
    ]
)

In [17]:
hydrodynamic_data

<xarray.Dataset> Size: 107kB
Dimensions:      (TIME: 4465, STATION: 2)
Coordinates:
  * TIME         (TIME) datetime64[us] 36kB 2026-04-30T23:00:00 ... 2026-05-3...
  * STATION      (STATION) <U1 8B '1' '0'
    LAT          (STATION) float64 16B 52.47 52.47
    LON          (STATION) float64 16B 4.614 4.604
    EPSG         (STATION) <U9 72B 'EPSG:4258' 'EPSG:4258'
Data variables:
    Water level  (STATION, TIME) float64 71kB -0.4 -0.4 -0.41 ... -0.54 -0.57

In [18]:
hydrodynamic_data.to_netcdf('hydrodynamic_data_IJmuiden.nc')